## **Build an AI Math Assistant with LangChain Tool Calling**

### import libraries

In [1]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
import torch

d:\Projects\ai-math-assistant-with-LangChain-and-LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

In [3]:
device = 0 if torch.cuda.is_available() else -1

In [4]:
def load_llm(model_name=MODEL_NAME):
    
    try:
        if not model_name:
            raise ValueError("Model name is empty or none")
        

        pipeline = HuggingFacePipeline.from_model_id(
            model_id=model_name,
            task="text-generation",
            device=device,
            pipeline_kwargs={
                "temperature": 0,
                "max_new_tokens": 256,
                "do_sample": False,  # needed for temperature to actually take effect
                "return_full_text":False
            }
        ) 
        
        llm_model = ChatHuggingFace(llm=pipeline)
        
        return llm_model
        
    except ValueError as e:
        print(f"Value error; {e}")
        raise
        
    except Exception as e:
        print(f"Error in load llm: {e}")
        raise

In [5]:
llm_model = load_llm()

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 4514.47it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [6]:
#llm_model.invoke("Hi how are you?")

Method 01:

Tool Calling with Langchain agent

In [7]:
def add_numbers(input:str)-> dict:
    
    """
    Add list of numbers provided in list of numbers or exracted from a string given and return their sum as
    {"result": total}.

    """
    
    numbers = [int(x) for x in input.replace(",","").split() if x.isdigit()]
    return {
        "result": sum(numbers)
    }

In [8]:
from langchain.agents import Tool

In [9]:
add_numbers_tool = Tool(
    name="AddTool",
    func=add_numbers,
    description="Adds a list of numbers and returns the result."
)

print("AddTool object: ", add_numbers_tool)

AddTool object:  name='AddTool' description='Adds a list of numbers and returns the result.' func=<function add_numbers at 0x000001EBF3C1B9C0>


In [10]:
print("Tool name:")
print(add_numbers_tool.name)

print("\nTool description:")
print(add_numbers_tool.description)

print("\nTool function")
print(add_numbers_tool.invoke)

Tool name:
AddTool

Tool description:
Adds a list of numbers and returns the result.

Tool function
<bound method BaseTool.invoke of Tool(name='AddTool', description='Adds a list of numbers and returns the result.', func=<function add_numbers at 0x000001EBF3C1B9C0>)>


In [11]:
test_input = "10 20 30 a b" 
print("result: ", add_numbers_tool.invoke(test_input))

result:  {'result': 60}


### `@tool` operator

Now you know how to create a tool with a `Tool` class (using Tool Interface), there's actually another way that you can create a tool using `@tool` decorator. The recommended way to create tools is using the `@tool` decorator. This decorator is designed to simplify the process of tool creation and should be used in most cases. After defining a function, you can decorate it with `@tool` to create a tool that implements the Tool Interface.

`@tool` opertor makes tools out of functions. See below:


In [12]:
from langchain_core.tools import tool

In [13]:
@tool
def add_numbers(input:str) -> dict:
    
    """
    take numbers from provided number list or extracted from string given and return dict type output for 
    sum of numbers like: {"result": total}
    """
    
    numbers = [int(x) for x in input.replace(",", "").split() if x.isdigit()]
    
    return{
        "result": sum(numbers)
    }

In [14]:
test_input = "what is the sum between 10, 20 and 30 " 
print(add_numbers.invoke(test_input))

{'result': 60}


## Zero-Shot-React-Description Agent

we need to define a tool that take and return plan string as inputs and outputs.

In [15]:
from langchain.agents import initialize_agent, AgentType

In [16]:
add_agent = initialize_agent(
    tools=[add_numbers_tool],
    llm=llm_model,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

C:\Users\Vish\AppData\Local\Temp\ipykernel_5660\3592493889.py:1: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  add_agent = initialize_agent(


In [17]:
test= "In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total."

In [18]:
# response = add_agent.run(
#     test
# )

In [20]:
@tool
def add_numbers_with_option(numbers:list[float], absolute:bool=False) -> float:
    
    """
        Add a list of numbers provided as input.
        
        parameters:
        - numbers (list[float]): a list of numbers to be summed.
        - absolute (bool): if True, get thr absolute values of the numbers before sum.
        
        Return:
        - float: the total sum of numbers
    """
    if absolute:
        numbers = [abs(n) for n in numbers]
    return sum(numbers)

In [21]:
add_numbers_with_option.args

{'numbers': {'items': {'type': 'number'}, 'title': 'Numbers', 'type': 'array'},
 'absolute': {'default': False, 'title': 'Absolute', 'type': 'boolean'}}

In [22]:
add_numbers_with_option.invoke({"numbers":[-1.1,2.65,32.5,-4.56], "absolute":False})

29.49

In [23]:
add_numbers_with_option.invoke({"numbers":[-1.1,2.65,32.5,-4.56], "absolute":True})

40.81